# KHOEM LoRA: វាស់មុន → បង្ហាត់ → វាស់ក្រោយ (Kaggle និង Colab)

notebook តែមួយ រកឃើញ platform ដោយខ្លួនឯង។
**Kaggle:** Settings → Accelerator = GPU T4, Internet = On, Add Input = dataset ដែលមាន train.jsonl និង valid.jsonl
**Colab:** Runtime → Change runtime type → T4 GPU → Save (ឯកសារត្រូវ upload ម្តងទៀតពេល runtime restart)

In [ ]:
# ០. ពិនិត្យ platform និង GPU (ត្រូវតែឆ្លងមុន)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth ប្រើ GPU តែមួយ
ON_KAGGLE = os.path.exists("/kaggle/input")
try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
import torch
assert torch.cuda.is_available(), "គ្មាន GPU! ជ្រើស Accelerator = GPU T4 រួចរត់ម្តងទៀត។"
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other", "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q unsloth datasets

In [ ]:
# ការកំណត់ (កែតែត្រង់នេះ)
MODEL   = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"

MAX_SEQ = 2048
N_EVAL  = 164          # ចំនួនបញ្ហា HumanEval (164 = ទាំងអស់ ប៉ុន្តែយឺត)
EPOCHS  = 1
LR      = 1e-4
RANK    = 8
MAX_TRAIN_ROWS = 4000 # សាកល្បងមុន; None = ប្រើទាំងអស់ (៣៩ ពាន់ ចំណាយពេលច្រើន)
SMOKE_MIX = {"python": 25, "rust": 20, "c": 16, "cpp": 14, "java": 11, "go": 11, "javascript": 2, "typescript": 1}  # % ក្នុង smoke (សរុប 100)
assert sum(SMOKE_MIX.values()) == 100
N_TEST = None        # None = test.jsonl ទាំងអស់ (4,012); ដាក់លេខតូចបើយឺត

## ១. ទិន្នន័យ (train.jsonl, valid.jsonl) និងការត្រួតពិនិត្យ

In [ ]:
import glob, shutil, json

if ON_KAGGLE:
    for n in ["train", "valid", "test"]:
        c = glob.glob(f"/kaggle/input/**/{n}*.jsonl", recursive=True)
        assert c, f"រក {n}.jsonl មិនឃើញ: Add Input → dataset ដែលមាន {n}.jsonl"
        shutil.copy(max(c, key=os.path.getmtime), f"{n}.jsonl")
else:
    from google.colab import files
    files.upload()   # ជ្រើស train.jsonl, valid.jsonl និង test.jsonl
    for n in ["train", "valid", "test"]:
        c = glob.glob(f"{n}*.jsonl")
        assert c, f"រក {n}.jsonl មិនឃើញ"
        newest = max(c, key=os.path.getmtime)
        if newest != f"{n}.jsonl":
            shutil.copy(newest, f"{n}.jsonl")

def to_msgs(ex):
    if "messages" in ex:
        return ex["messages"]
    for a, b in [("instruction","output"), ("prompt","completion"), ("prompt","response"), ("input","output"), ("question","answer")]:
        if a in ex and b in ex:
            return [{"role":"user","content":ex[a]}, {"role":"assistant","content":ex[b]}]
    raise KeyError("unknown columns: " + str(list(ex.keys())))

def load(f):
    return [json.loads(l) for l in open(f, encoding="utf8") if l.strip()]

def ans(ex):
    return to_msgs(ex)[-1]["content"]

tr, va = load("train.jsonl"), load("valid.jsonl")
va_ans = {ans(x) for x in va}
leak = sum(ans(x) in va_ans for x in tr)
print(f"train={len(tr)}  valid={len(va)}  train rows ដែលចម្លើយនៅក្នុង valid={leak}")
assert leak == 0, "មាន leak: train/valid ជា read-only ក្នុង notebook ត្រូវកែនៅខាងក្រៅ"
te = load("test.jsonl")
te_ans = {ans(x) for x in te}
leak_te = sum(ans(x) in te_ans for x in tr) + sum(ans(x) in te_ans for x in va)
print(f"test={len(te)}  leak test↔train/valid={leak_te}")
assert leak_te == 0, "test ត្រូវដាច់ពី train/valid"
from collections import Counter
print("lang train:", Counter(x.get("lang", "?") for x in tr).most_common(8))
print("lang valid:", Counter(x.get("lang", "?") for x in va).most_common(8))

## ២. ផ្ទុក model + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True,
)
LORA_ADDED = False   # កុំដាក់ LoRA ពីរដង
print("model ready")

## ៣. វាស់ HumanEval (pass@1, greedy) ព្រមទាំងបង្ហាញចម្លើយគំរូ

In [ ]:
import re, subprocess, tempfile, warnings, time
from datasets import load_dataset
warnings.filterwarnings("ignore")
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

try:
    he = load_dataset("openai/openai_humaneval", split="test")
except Exception:
    he = load_dataset("openai_humaneval", split="test")
he = he.select(range(N_EVAL))

def extract(text):
    m = re.search(r"```[A-Za-z0-9_+-]*\n(.*?)```", text, flags=re.S)
    return m.group(1) if m else text

def run_eval(model, show=2):
    FastLanguageModel.for_inference(model)
    passed = 0
    for k, ex in enumerate(he):
        msgs = [{"role": "user", "content":
                 "Complete this Python function. Reply with the full function "
                 "in a single fenced python code block.\n\n" + ex["prompt"]}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
        if not torch.is_tensor(ids):
            ids = ids["input_ids"]
        ids = ids.to("cuda")
        with torch.no_grad():
            out = model.generate(input_ids=ids, max_new_tokens=512, do_sample=False)
        text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        body = extract(text)
        if f"def {ex['entry_point']}" not in body:
            body = ex["prompt"] + body
        prog = body + "\n\n" + ex["test"] + f"\ncheck({ex['entry_point']})\n"
        with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
            f.write(prog); path = f.name
        ok = False
        try:
            r = subprocess.run(["python", path], capture_output=True, timeout=10)
            ok = (r.returncode == 0)
        except subprocess.TimeoutExpired:
            pass
        finally:
            os.unlink(path)
        passed += ok
        if k < show:
            print(f"--- គំរូ {k}: {'PASS' if ok else 'FAIL'} ---")
            print(text[:350])
    return passed / len(he)

assert not LORA_ADDED, "baseline ត្រូវវាស់មុនដាក់ LoRA: Restart រួចរត់ពី cell 1"
_t0 = time.time()
before = run_eval(model)
_he_secs = time.time() - _t0
print(f"HumanEval មុន: {before:.3f}  ({N_EVAL} បញ្ហា)")
print(f"HumanEval {N_EVAL} បញ្ហា ប្រើពេល {_he_secs:.0f}s")

def test_loss(model, n=None):
    FastLanguageModel.for_inference(model)
    rows = te if n is None else te[:n]
    tot = cnt = skipped = 0
    t0 = time.time()
    for ex in rows:
        msgs = to_msgs(ex)
        ids = []
        for m, kw in ((msgs[:-1], dict(add_generation_prompt=True)), (msgs, {})):
            x = tokenizer.apply_chat_template(m, return_tensors="pt", **kw)
            ids.append(x if torch.is_tensor(x) else x["input_ids"])
        plen = ids[0].shape[1]
        full = ids[1][:, :MAX_SEQ].to("cuda")
        if full.shape[1] <= plen:
            skipped += 1
            continue
        with torch.no_grad():
            lg = model(input_ids=full).logits
        assert lg.shape[1] == full.shape[1], "logits មិនត្រូវនឹង input"
        loss = torch.nn.functional.cross_entropy(lg[0, plen - 1:-1].float(), full[0, plen:], reduction="sum")
        tot += loss.item(); cnt += full.shape[1] - plen
    print(f"test rows={len(rows) - skipped} (រំលង {skipped}) tokens={cnt} | {time.time() - t0:.0f}s")
    return tot / cnt

tl_before = test_loss(model, N_TEST)
print(f"KHOEM test loss មុន: {tl_before:.4f}")

## ៤. បង្ហាត់ LoRA

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

assert not LORA_ADDED, "LoRA ត្រូវបានដាក់រួច: Restart រួចរត់ពី cell 1"
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=RANK * 2, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

LORA_ADDED = True
FastLanguageModel.for_training(model)

def fmt(ex):
    return {"text": tokenizer.apply_chat_template(to_msgs(ex), tokenize=False)}

train = load_dataset("json", data_files="train.jsonl", split="train")
valid = load_dataset("json", data_files="valid.jsonl", split="train")
if MAX_TRAIN_ROWS:
    import random
    rng = random.Random(3407)
    by_lang = {}
    for i, lg in enumerate(train["lang"]):
        by_lang.setdefault(lg, []).append(i)
    pick = []
    for lg, pct in SMOKE_MIX.items():
        k = MAX_TRAIN_ROWS * pct // 100
        assert len(by_lang.get(lg, [])) >= k, f"{lg}: មិនគ្រប់ {k} ជួរ"
        pick += rng.sample(by_lang[lg], k)
    train = train.select(sorted(pick)).shuffle(seed=3407)
    print("smoke mix:", {lg: MAX_TRAIN_ROWS * pct // 100 for lg, pct in SMOKE_MIX.items()})
else:
    print(f"full train: {len(train):,} rows")
train = train.map(fmt)
valid = valid.map(fmt)
print(f"Dataset: {len(train):,} train / {len(valid):,} valid  (~{len(train)//8} ជំហាន)")

bf16 = torch.cuda.is_bf16_supported()   # T4 = False -> ប្រើ fp16
sig = inspect.signature(SFTConfig.__init__).parameters
params = dict(
    output_dir="outputs", per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, num_train_epochs=EPOCHS,
    learning_rate=LR, warmup_steps=5, logging_steps=10, report_to="none",
    dataset_text_field="text", optim="adamw_8bit",
    fp16=not bf16, bf16=bf16, seed=3407,
)
params["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "epoch"
if "max_length" in sig: params["max_length"] = MAX_SEQ
elif "max_seq_length" in sig: params["max_seq_length"] = MAX_SEQ

kwargs = dict(model=model, train_dataset=train, eval_dataset=valid, args=SFTConfig(**params))
tsig = inspect.signature(SFTTrainer.__init__).parameters
kwargs["processing_class" if "processing_class" in tsig else "tokenizer"] = tokenizer

trainer = SFTTrainer(**kwargs)
trainer.train()

## ៥. វាស់ម្តងទៀត

In [ ]:
after = run_eval(model)
print(f"HumanEval មុន: {before:.3f}  →  ក្រោយ: {after:.3f}")
SAVE_OK = after >= before - 0.03
if not SAVE_OK:
    print("⚠️ ពិន្ទុធ្លាក់ច្រើន: កុំប្រើ adapter នេះ។ មើលចម្លើយគំរូខាងលើ ថាខុសទម្រង់ ឬខុសសមត្ថភាព។")
tl_after = test_loss(model, N_TEST)
print(f"KHOEM test loss: {tl_before:.4f} → {tl_after:.4f}  ({'ចុះ ល្អឡើង' if tl_after < tl_before else 'មិនចុះ'})")
SMOKE = bool(MAX_TRAIN_ROWS)
print("Gate:", "ឆ្លង ✅" if SAVE_OK else "មិនឆ្លង ❌", "|", "smoke (មិនរក្សាទុក)" if SMOKE else "full (រក្សាទុកបើឆ្លង)")

## ៦. រក្សាទុក (តែពេលពិន្ទុមិនធ្លាក់)

In [ ]:
if SMOKE:
    print(
        "smoke ចប់: មិនរក្សាទុក។",
        "ឆ្លង gate ✅ → កែ MAX_TRAIN_ROWS=None ក្នុង cell 3 រួច Restart រត់ពី cell 1 (Full Train)"
        if SAVE_OK else
        "មិនឆ្លង gate ❌ → កុំ Full Train"
    )
else:
    assert SAVE_OK, "ពិន្ទុធ្លាក់ មិនរក្សាទុក"
    model.save_pretrained("khoem-lora")
    tokenizer.save_pretrained("khoem-lora")
    OUT = "/kaggle/working" if ON_KAGGLE else "."
    zp = shutil.make_archive(os.path.join(OUT, "khoem-lora"), "zip", "khoem-lora")
    print("zip:", zp)
    if ON_COLAB and not ON_KAGGLE:
        from google.colab import files
        files.download(zp)
    else:
        print("Kaggle: ទាញយកពីផ្ទាំង Output (ខាងស្តាំ) ក្រោយ Save Version")